# Synthetic Data Generation: From Seed Image to Augmented Video

This notebook demonstrates how generative models can scale a single seed image into a large, diverse set of training videos — without additional real-world data collection.


```
Seed Image  ──►  Image Editing (I2I)  ──►  Video Generation (I2V)  ──►  Video Transfer (V2V)
               (Qwen Image Edit)          (Cosmos3)                    (Cosmos3 Transfer)
```


**Running example:** This notebook uses a traffic intersection camera frame as its seed image to make the stages concrete. The same pipeline applies to any domain — robotics, warehouses, retail, smart spaces — by swapping the seed image and prompts.

**Terminology used in this notebook:**
- **I2I — Image-to-Image**: modifying an existing image based on a text instruction; appearance changes, structure and content are preserved
- **T2I — Text-to-Image**: generating a brand-new image from a text description alone, with no input image
- **I2V — Image-to-Video**: generating a video clip from a still image, using it to anchor the first frame
- **V2V — Video-to-Video**: re-rendering a video's visual appearance while preserving its coarse structure and motion

**What each stage contributes:**

| Stage | Main variation introduced | Intended invariants |
|---|---|---|
| I2I | Scene appearance (lighting, weather, style) | Composition and subject identity |
| I2V | Temporal behavior (motion, events) | Starting image and scene context |
| V2V | Video appearance (materials, environment) | Coarse structure and motion |

Note that these invariants reflect the *intended conditioning behavior* — generated outputs can drift, hallucinate, or exhibit temporal artifacts. Always review outputs before using them downstream.

Each stage multiplies data diversity along a different axis:
- **Stage 1 (I2I)** varies appearance at the image level — lighting, style, conditions
- **Stage 2 (I2V)** adds a temporal dimension — turning any image into a video clip
- **Stage 3 (V2V)** re-renders the video's visual properties while preserving structure and motion

### Learning Objectives

By the end of this notebook you will be able to:
- Understand how generative models enable dataset scaling from a single image to many diverse training videos
- Identify what each pipeline stage contributes and why the combination produces greater diversity than any single stage alone
- Control each model's output through prompts and parameters to target specific types of variation useful for training
- Use an LLM to assist in prompt generation for the video generation stage
- Reason about the compound scaling effect across stages and apply it to estimate synthetic dataset size

### Table of Contents

**[Part 0: Seed Image](#Part-0:-Seed-Image)**  
**[Part 1: Image Augmentation with Qwen](#Part-1:-Image-Augmentation-with-Qwen)**  
&nbsp;&nbsp;&nbsp;&nbsp;[1.1 Load Pipeline](#11-Load-Pipeline)  
&nbsp;&nbsp;&nbsp;&nbsp;[1.2 Craft an Edit Prompt](#12-Craft-an-Edit-Prompt)  
&nbsp;&nbsp;&nbsp;&nbsp;[1.3 Run Augmentation](#13-Run-Augmentation)  
**[Part 2: Image-to-Video Generation with Cosmos](#Part-2:-Image-to-Video-Generation-with-Cosmos)**  
&nbsp;&nbsp;&nbsp;&nbsp;[2.1 Environment Setup](#21-Environment-Setup)  
&nbsp;&nbsp;&nbsp;&nbsp;[2.2 Structured Prompts](#22-Structured-Prompts)  
&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;[2.2.1 Generate a Prompt with AI (Optional)](#221-Generate-a-Prompt-with-AI-Optional)  
&nbsp;&nbsp;&nbsp;&nbsp;[2.3 Run Image-to-Video Inference](#23-Run-Image-to-Video-Inference)  
**[Part 3: Video Augmentation with Cosmos](#Part-3:-Video-Augmentation-with-Cosmos)**  
&nbsp;&nbsp;&nbsp;&nbsp;[3.1 Transfer Controls](#31-Transfer-Controls)  
&nbsp;&nbsp;&nbsp;&nbsp;[3.2 Run Edge Transfer](#32-Run-Edge-Transfer)  
**[Review](#Review)**

---
## Part 0: Seed Image

The seed image is the anchor for your entire SDG pipeline. It defines the scene, camera viewpoint, and visual context that all downstream stages inherit.

Place your own image in `assets/inputs/` and update `SEED_IMAGE_NAME`. The default used here is a frame captured from a live overhead intersection camera.

**What makes a good seed image:**
- Clear, well-lit subject with a defined viewpoint
- Representative of your target deployment conditions
- High enough resolution for generation quality (480p or higher is ideal)
- Can be real (captured) or synthetic (rendered) — both work

Working directory paths are configured here, then your image is loaded for display.

In [ ]:
from pathlib import Path
from PIL import Image
from IPython.display import display

SDG_ROOT = Path.cwd()  # assumes notebook is run from SDG_Part1/
ASSETS = SDG_ROOT / "assets"
INPUTS = ASSETS / "inputs"
OUTPUTS = ASSETS / "outputs"
OUTPUTS.mkdir(parents=True, exist_ok=True)

SEED_IMAGE_NAME = "site03_median.jpg"
seed_image_path = INPUTS / SEED_IMAGE_NAME

if seed_image_path.exists():
    seed_image = Image.open(seed_image_path).convert("RGB")
    print(f"Loaded: {seed_image_path} | size: {seed_image.size}")
else:
    print(f"No image found at {seed_image_path}")
    print(f"Add your image to assets/inputs/ and set SEED_IMAGE_NAME in the Configuration cell.")
    seed_image = Image.new("RGB", (1280, 720), color=(90, 110, 130))
    INPUTS.mkdir(parents=True, exist_ok=True)
    seed_image.save(seed_image_path)

display(seed_image)

---
## Part 1: Image Augmentation with Qwen

Qwen Image Edit is an instruction-following image-to-image (I2I) model. Given an input image and a natural language instruction, it modifies the image according to the instruction while leaving unspecified attributes largely unchanged. In practice, outputs can still shift in unintended ways — particularly with strong edits — so reviewing the result is important.

In an SDG context, I2I augmentation is used to:
- Vary environmental conditions (lighting, weather, time of day)
- Change material or surface properties
- Add or remove scene elements
- Simulate sensor or camera variations

The model operates on the full image without requiring a mask — the instruction determines what changes.

### 1.1 Load Pipeline

This cell loads the Qwen model weights into GPU memory. This takes about 30 seconds and only needs to run once per session — all subsequent edits reuse the loaded model.

In [ ]:
import sys, importlib, subprocess
import torch

for pkg in ["diffusers", "transformers", "accelerate", "Pillow"]:
    mod = "PIL" if pkg == "Pillow" else pkg
    try:
        importlib.import_module(mod)
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

from diffusers import QwenImageEditPipeline

DEVICE = "cuda:3"
DTYPE = torch.bfloat16

print(f"Loading Qwen Image Edit on {DEVICE} ({DTYPE})...")
qwen_pipeline = QwenImageEditPipeline.from_pretrained("Qwen/Qwen-Image-Edit", torch_dtype=DTYPE)
qwen_pipeline.to(DEVICE)
print("Pipeline ready.")

### 1.2 Craft an Edit Prompt

Qwen Image Edit uses natural language instructions. Unlike text-to-image (T2I) prompts that describe the target from scratch, I2I prompts describe the *change* relative to the input image — what should be different, and what should stay the same.

**Prompt structure that works well:**

| Component | Purpose | Example |
|---|---|---|
| **What to change** | Explicit instruction | `"Change the lighting to..."` |
| **What to preserve** | Prevents unintended drift | `"Keep all objects and their positions unchanged"` |
| **Target appearance** | Specific, not vague | `"...overcast with diffuse shadows"` not `"...different"` |

**Parameter guide:**
- `true_cfg_scale` 3–5: lower = more creative, higher = stricter prompt adherence
- `num_inference_steps` 30–50: more steps = higher quality, slower

Set your parameters and prompt below. Running it stores your configuration — no editing starts yet.

In [ ]:
# --- Configure your edit here ---
EDIT_PROMPT = "Change the scene to a rainy morning. Add wet road surfaces with reflections, overcast sky, and light rain. Keep all scene elements, objects, and their positions exactly as they are."

EDIT_PARAMS = {
    "true_cfg_scale": 4.0,
    "num_inference_steps": 50,
    "seed": 42,
}
# ---------------------------------

print(f"Prompt: {EDIT_PROMPT}")
print(f"Params: {EDIT_PARAMS}")

### 1.3 Run Augmentation

Next step is to send your seed image and instruction to Qwen and wait for the edited result. Expect 1–2 minutes on a single GPU. The output is saved to a timestamped folder under `assets/outputs/i2i/` along with a metadata file recording the prompt and parameters used.

In [ ]:
import json
from pathlib import Path
from PIL import Image
from datetime import datetime

# Re-establish paths in case this is a fresh kernel session
# Run the Seed Image cell first if SEED_IMAGE_NAME is not defined
SDG_ROOT = Path.cwd()
ASSETS = SDG_ROOT / "assets"
INPUTS = ASSETS / "inputs"
OUTPUTS = ASSETS / "outputs"
OUTPUTS.mkdir(parents=True, exist_ok=True)

seed_image_path = INPUTS / SEED_IMAGE_NAME
seed_image = Image.open(seed_image_path).convert("RGB")

# Each run gets its own timestamped folder
RUN_I2I = datetime.now().strftime("%Y%m%d_%H%M%S")
i2i_run_dir = OUTPUTS / "i2i" / RUN_I2I
i2i_run_dir.mkdir(parents=True, exist_ok=True)

with torch.inference_mode():
    result = qwen_pipeline(
        image=seed_image,
        prompt=EDIT_PROMPT,
        negative_prompt=" ",
        generator=torch.manual_seed(EDIT_PARAMS["seed"]),
        true_cfg_scale=EDIT_PARAMS["true_cfg_scale"],
        num_inference_steps=EDIT_PARAMS["num_inference_steps"],
    )

augmented_image = result.images[0]
augmented_image_path = i2i_run_dir / "augmented_image.png"
augmented_image.save(augmented_image_path)

meta = {"stage": "i2i", "run": RUN_I2I, "prompt": EDIT_PROMPT, "params": EDIT_PARAMS,
        "source": str(seed_image_path), "output": str(augmented_image_path)}
(i2i_run_dir / "meta.json").write_text(json.dumps(meta, indent=2))
print(f"Run: {RUN_I2I}")
print(f"Saved: {augmented_image_path}")

The cell below displays the seed image and the edited result side by side so you can compare what changed.

**After running, here is what to look for:**
- Did the requested appearance change occur?
- Did the scene composition, subject identity, and layout remain unchanged?
- Did anything shift unexpectedly that you did not ask to change?
- Is this a usable augmented image, or would you adjust the prompt and regenerate?

In [ ]:
import base64, io
from IPython.display import HTML, display

def image_to_b64(img):
    buf = io.BytesIO()
    img.save(buf, format="PNG")
    return base64.b64encode(buf.getvalue()).decode()

display(HTML(f"""
<div style="display: flex; gap: 20px;">
  <div style="flex: 1;">
    <h3>Seed Image</h3>
    <img src="data:image/png;base64,{image_to_b64(seed_image)}" style="max-width: 100%;">
  </div>
  <div style="flex: 1;">
    <h3>Augmented (Qwen Image Edit)</h3>
    <img src="data:image/png;base64,{image_to_b64(augmented_image)}" style="max-width: 100%;">
    <p style="font-size: 12px; font-family: monospace;">Prompt: {EDIT_PROMPT}</p>
  </div>
</div>
"""))

If the result doesn't meet your expectations, go back to section 1.2, update `EDIT_PROMPT` or `EDIT_PARAMS`, and rerun the augmentation cell. Each run is saved to a timestamped folder under `assets/outputs/i2i/` so all previous results are preserved.

---
## Part 2: Image-to-Video Generation with Cosmos

Cosmos3 is a World Foundation Model(WFM) that generates video from a seed image. Image-to-video (I2V) generation is conditioned on the provided image to anchor the first frame, then synthesizes motion forward in time consistent with the scene geometry and the action described in the prompt. The model aims to produce physically plausible motion, though outputs may exhibit temporal artifacts, drift from the seed image, or inconsistencies — especially over longer clips.

I2V adds a temporal dimension to your image dataset: one image can yield multiple distinct video clips by varying the motion prompt or the random seed.

**Optional — free Qwen from GPU memory before running Cosmos.**  
Both models share one GPU. If Cosmos hits an out-of-memory error in Part 2, run this first to unload Qwen and recover the memory.

In [ ]:
import gc
del qwen_pipeline
gc.collect()
torch.cuda.empty_cache()
print(f"Qwen unloaded. GPU free memory: {torch.cuda.mem_get_info(3)[0] / 1024**3:.1f} GiB")

### 2.1 Environment Setup

The Cosmos Framework is **pre-installed** in this lab at `/opt/cosmos3-framework` (its virtual environment is the kernel you are using), so there is no clone or `uv sync` step. The cell below points the notebook at that checkout and the cookbook's sample assets, and selects the GPU.

This notebook uses **Cosmos3-Nano** by default — a 16B parameters single-GPU model optimized for fast iteration. **Cosmos3-Super** (32B parameters, requires 2 GPUs) produces significantly higher visual quality and better motion realism, and is the recommended choice for final dataset generation. To switch, set `COSMOS3_CHECKPOINT_PATH=Cosmos3-Super`, `CUDA_VISIBLE_DEVICES=0,1`, and `COSMOS3_NUM_GPUS=2` before running the inference cells.

Each inference run in Parts 2 and 3 saves its outputs to a timestamped subfolder so nothing is overwritten:
- `assets/outputs/i2v/YYYYMMDD_HHMMSS/` — generated video, payload, and metadata
- `assets/outputs/transfer_edge/YYYYMMDD_HHMMSS/` — transferred video, spec, and metadata


In [ ]:
import os, json, socket
from pathlib import Path

def free_local_port():
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
        s.bind(("127.0.0.1", 0))
        return str(s.getsockname()[1])

#Set your paths here if running locally. 
# COSMOS_ROOT = os.environ["COSMOS_ROOT"]  = "/localhome/local-sisimbimanzi/cosmos"
# COSMOS3_REPO = os.environ["COSMOS3_REPO"] = "/localhome/local-sisimbimanzi/cosmos/packages/cosmos-framework"

COSMOS_ROOT = Path(os.environ.get("COSMOS_ROOT", "/localhome/local-sisimbimanzi/cosmos")).resolve()
COSMOS3_REPO = Path(os.environ.get("COSMOS3_REPO", "/localhome/local-sisimbimanzi/cosmos/packages/cosmos-framework")).resolve()

# Uncomment the following lines if running in a DLI environment for the pre-installed framework and cookbooks.
# COSMOS_ROOT = Path(os.environ.get("COSMOS_ROOT", "/dli/task/cosmos")).resolve()
# COSMOS3_REPO = Path(os.environ.get("COSMOS3_REPO", "/opt/cosmos3-framework")).resolve()


COSMOS3_VENV = COSMOS3_REPO / ".venv"
AUDIOVISUAL_ROOT = COSMOS_ROOT / "cookbooks" / "cosmos3" / "generator" / "audiovisual"
TRANSFER_ROOT = COSMOS_ROOT / "cookbooks" / "cosmos3" / "generator" / "transfer"

# SDG notebook paths
SDG_ROOT = Path.cwd()
ASSETS = SDG_ROOT / "assets"
INPUTS = ASSETS / "inputs"
OUTPUTS = ASSETS / "outputs"
CONFIGS = ASSETS / "configs"
for d in [OUTPUTS, CONFIGS]:
    d.mkdir(parents=True, exist_ok=True)

# Input image: use augmented_image_path from Part 1 if still in scope,
# otherwise find the most recent I2I run, then fall back to seed image
try:
    AUGMENTED_IMAGE = augmented_image_path
    print(f"Using augmented image from current session: {AUGMENTED_IMAGE}")
except NameError:
    i2i_outputs = sorted((OUTPUTS / "i2i").rglob("augmented_image.png")) if (OUTPUTS / "i2i").exists() else []
    if i2i_outputs:
        AUGMENTED_IMAGE = i2i_outputs[-1]
        print(f"Using latest I2I run: {AUGMENTED_IMAGE}")
    else:
        AUGMENTED_IMAGE = INPUTS / SEED_IMAGE_NAME   # falls back to seed image
        print(f"NOTE: no augmented image found — using seed image: {AUGMENTED_IMAGE}")

# ── GPU configuration ─────────────────────────────────────────────────────────
os.environ["CUDA_VISIBLE_DEVICES"] = "3"
os.environ.setdefault("COSMOS3_NUM_GPUS", "1")
# ─────────────────────────────────────────────────────────────────────────────

os.environ.setdefault("COSMOS3_MASTER_ADDR", "127.0.0.1")
os.environ.setdefault("COSMOS3_MASTER_PORT", free_local_port())
os.environ.setdefault("HF_HOME", str(Path.home() / ".cache" / "huggingface"))
os.environ["COSMOS3_REPO"] = str(COSMOS3_REPO)
COSMOS3_CHECKPOINT = os.environ.get("COSMOS3_CHECKPOINT_PATH", "Cosmos3-Nano")
os.environ["COSMOS3_CHECKPOINT_PATH"] = COSMOS3_CHECKPOINT

for label, path in [("Cosmos root", COSMOS_ROOT), ("Cosmos Framework", COSMOS3_REPO), ("Audiovisual assets", AUDIOVISUAL_ROOT)]:
    status = "OK" if path.exists() else "MISSING — check Configuration cell"
    print(f"{label}: {path} [{status}]")
print(f"Input image:  {AUGMENTED_IMAGE} [{'OK' if AUGMENTED_IMAGE.exists() else 'MISSING'}]")
print(f"Checkpoint:   {COSMOS3_CHECKPOINT}")
print(f"CUDA devices: {os.environ['CUDA_VISIBLE_DEVICES']} | GPUs: {os.environ['COSMOS3_NUM_GPUS']}")


Confirm the GPU is visible to the Cosmos kernel before proceeding. If `cuda available: False`, the inference cells will fail — check your kernel selection and GPU allocation.

In [ ]:
import torch
print(f"torch: {torch.__version__} | cuda: {torch.version.cuda}")
print(f"cuda available: {torch.cuda.is_available()} | device count: {torch.cuda.device_count()}")
for i in range(torch.cuda.device_count()):
    print(f"  device {i}: {torch.cuda.get_device_name(i)}")

### 2.2 Structured Prompts

Cosmos uses **structured JSON prompts** rather than a single string. The structure separates scene description into semantic components, giving the model stronger grounding on what to generate.

The key fields:

| Field | What it controls |
|---|---|
| `subjects[].description` | What the main element(s) are |
| `subjects[].action` | **The motion** — what happens during the clip |
| `subjects[].appearance_details` | Surface, color, texture details to preserve |
| `subjects[].orientation` / `pose` / `number_of_subjects` | Additional per-subject grounding |
| `background_setting` | Environment, geometry, first-frame anchor instruction |
| `lighting` | Illumination conditions, shadows, atmosphere |
| `aesthetics` | Composition, color scheme, mood, patterns |
| `cinematography` | Camera motion, framing, angle, and first-frame preservation |

**For I2V specifically:**
- The `action` field is the most important — it is the primary driver of what the model generates
- Include `"Use the provided image as the exact first frame"` in `background_setting` and `cinematography.camera_preservation` to signal the anchor
- Describe *motion*, not static appearance — the image already provides appearance context
- `cinematography.camera_motion` should state the camera is fixed/locked unless a motion-control checkpoint is being used

The full schema — including the `aesthetics`, `cinematography`, and `negative_prompt` fields — is defined in [`assets/prompts/i2v_prompt_schema.json`](assets/prompts/i2v_prompt_schema.json) rather than inline here, since it's long. The prompt-generation cell below (2.2.1) loads it directly.

Three traffic-specific scenario prompts are provided in `assets/prompts/` as a concrete example of this structure applied to the intersection seed image.

### 2.2.1 Generate a Prompt with AI

Instead of writing the structured JSON prompt by hand, you can use an NVIDIA-hosted multimodal model to generate one from your seed image and a plain-language description of the action you want. This requires an `NVIDIA_API_KEY` set in your environment.

**Model: `nvidia/llama-3.1-nemotron-nano-vl-8b-v1`** — a compact (8B) multimodal vision-language model built on Llama 3.1 and post-trained by NVIDIA (part of the Nemotron family). It accepts both an image and a text prompt in the same request and returns a text completion, which makes it well suited for lightweight image-grounded tasks like this one — reading a seed frame and turning a short action description into a structured prompt — without the cost of a larger model.

The model receives your seed image and your one-sentence description, and outputs a valid Cosmos structured JSON that you can plug directly into the inference step below.


In [ ]:
import sys, importlib, subprocess
for import_name, pip_name in [("openai", "openai"), ("dotenv", "python-dotenv")]:
    try:
        importlib.import_module(import_name)
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pip_name])

import os, json, base64, re
from pathlib import Path
from openai import OpenAI
from dotenv import load_dotenv

# Loads NVIDIA_API_KEY from the .env file.
# Edit that file and put your key there: NVIDIA_API_KEY=nvapi-...
_env_path = Path.cwd() / ".env"
_loaded = load_dotenv(_env_path, override=True)
NVIDIA_API_KEY = os.environ.get("NVIDIA_API_KEY", "")
print(f".env found at {_env_path}: {_env_path.exists()} | loaded: {_loaded} | key set: {bool(NVIDIA_API_KEY)}")
if not NVIDIA_API_KEY:
    raise EnvironmentError(
        "NVIDIA_API_KEY is not set. Edit the .env file and put your key there: NVIDIA_API_KEY=nvapi-... "
        "then restart the kernel or re-run this cell."
    )

# --- Configure ---
VIDEO_DESCRIPTION = "Multiple vehicles driving through the scene in a normal traffic situation, while abiding to traffic lights to avoid collision." # change this to your desired scenario
USE_VISION = True  
# -----------------

_sdg_root = Path.cwd()
_assets = _sdg_root / "assets"
_inputs = _assets / "inputs"
_seed_image_path = _inputs / SEED_IMAGE_NAME

print(f"Description: {VIDEO_DESCRIPTION}")
print(f"Seed image:  {_seed_image_path} [{'found' if _seed_image_path.exists() else 'NOT FOUND'}]")

This sends your image and description to the multimodal model, which streams back the final JSON prompt — typically takes a few seconds.

In [ ]:
# I2V prompts follow a richer schema — see assets/prompts/i2v_prompt_schema.json
COSMOS_PROMPT_SCHEMA_FILE = ASSETS / "prompts" / "i2v_prompt_schema.json"
COSMOS_PROMPT_SCHEMA = COSMOS_PROMPT_SCHEMA_FILE.read_text()

SYSTEM_PROMPT = f"""You are an expert at writing structured prompts for the Cosmos3 world foundation model for image-to-video (I2V) generation.

Given a seed image (the exact first frame of the video to be generated) and a description of the desired action, produce a structured JSON prompt relatively following this format while giving enough description to help with video generation:

{COSMOS_PROMPT_SCHEMA}

Critical rules:
- Output ONLY valid JSON — no markdown fences, no preamble, no explanation after
- The "action" field is the most important: describe physically realistic, specific motion with direction, speed, and trajectory
- Include the first-frame anchor instruction in "background_setting" and "cinematography.camera_preservation" exactly as shown in the schema
- Describe motion, not static appearance — the image already defines appearance
- "cinematography.camera_motion" should state the camera is fixed/locked unless explicitly told otherwise
- Add additional subject entries if the description involves multiple moving elements"""

# Build message — include image if available and USE_VISION is True
if USE_VISION and _seed_image_path.exists():
    ext = _seed_image_path.suffix.lower().lstrip(".")
    mime = "image/jpeg" if ext in ("jpg", "jpeg") else f"image/{ext}"
    image_b64 = base64.b64encode(_seed_image_path.read_bytes()).decode()
    user_content = [
        {"type": "image_url", "image_url": {"url": f"data:{mime};base64,{image_b64}"}},
        {"type": "text", "text": f"Desired action: {VIDEO_DESCRIPTION}\n\nGenerate the Cosmos3 structured JSON prompt for this image."},
    ]
    print(f"Sending image ({_seed_image_path.name}) + description to Nemotron...")
else:
    if USE_VISION:
        print("NOTE: Seed image not found — sending text-only request.")
    user_content = f"Desired action: {VIDEO_DESCRIPTION}\n\nGenerate the Cosmos3 structured JSON prompt."
    print("Sending text-only request to Nemotron...")

client = OpenAI(
    base_url="https://integrate.api.nvidia.com/v1",
    api_key=NVIDIA_API_KEY,
)

response_text = ""
completion = client.chat.completions.create(
    model="nvidia/llama-3.1-nemotron-nano-vl-8b-v1",
    messages=[
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_content},
    ],
    temperature=0.6,
    top_p=0.95,
    max_tokens=4096,
    stream=True,
)

print("Generating... ", end="", flush=True)
for chunk in completion:
    if not chunk.choices:
        continue
    delta = chunk.choices[0].delta
    if delta.content:
        response_text += delta.content
        print(delta.content, end="", flush=True)

print("\n")

After the model responds, run this to parse the output as JSON, save it to `assets/prompts/llm_generated_prompt.json`, and render it for review. If parsing fails (the model added extra text), the raw output is saved for inspection.

In [ ]:
import json
import re
from IPython.display import JSON, display

# Strip markdown fences if the model wrapped the output
raw = response_text.strip()
raw = re.sub(r"^```(?:json)?\s*\n?", "", raw)
raw = re.sub(r"\n?```\s*$", "", raw)
raw = raw.strip()

try:
    generated_prompt = json.loads(raw)
    output_path = _assets / "prompts" / "llm_generated_prompt.json"
    output_path.write_text(json.dumps(generated_prompt, indent=2))
    print(f"Saved: {output_path}")
    print("\nTo use this prompt for generation, update SAMPLE_PROMPT_FILE in the inference config cell:")
    print('  SAMPLE_PROMPT_FILE = ASSETS / "prompts" / "llm_generated_prompt.json"')
    print()
    display(JSON(generated_prompt, expanded=True))
except json.JSONDecodeError as e:
    print(f"Could not parse response as JSON: {e}")
    print("The model may have included extra text. Raw output saved to:")
    raw_path = _assets / "prompts" / "llm_generated_prompt_raw.txt"
    raw_path.write_text(response_text)
    print(f"  {raw_path}")
    print("\nRaw response:\n", response_text)

Alternatively, three sample prompts are provided in `assets/prompts/`, each describing a different traffic scenario at the intersection. Choose the one that matches what you want to generate, then inspect it below before running inference.

In [ ]:
from IPython.display import JSON, display

SAMPLE_PROMPT_FILE = ASSETS / "prompts" / "i2v_normal_traffic.json"      #edit to point to your desired prompt

prompt_data = json.loads(SAMPLE_PROMPT_FILE.read_text())
display(JSON(prompt_data, expanded=True))

### 2.3 Run Image-to-Video Inference

**Image-to-video best practices:**

- **Keep `action` grounded in the image**: if your image shows a vehicle on a road, the action should describe vehicle motion — not flying or teleporting
- **Camera stays fixed**: Cosmos I2V uses a locked camera by default; don't describe camera motion unless using a motion-control checkpoint
- **`seed`**: fix it to reproduce results; vary it across runs to get motion diversity from the same prompt

**Key generation parameters:**

| Parameter | What it does | Typical range |
|---|---|---|
| `num_steps` | Number of denoising steps — more steps = higher quality, slower generation | 20–50 |
| `guidance` | How strictly the model follows the prompt vs. exploring freely — higher = closer to prompt | 4.0–8.0 |
| `shift` | Controls the noise schedule; affects motion sharpness and temporal consistency | 2.0–10.0 |
| `fps` | Frames per second of the output video | 24–30 |
| `num_frames` | Total frames generated — at 24 fps: 121 frames ≈ 5 s, 193 frames ≈ 8 s | 121–193 |
| `resolution` | Output resolution — `"480"` is faster to generate, `"720"` gives higher visual quality | `"480"` / `"720"` |


Two steps follow: first assemble the payload, then run generation. Generation takes a few minutes per clip on a single H100.

Run this cell to assemble the payload from the prompt file and input image, and preview what will be sent to the inference script.

In [ ]:
from IPython.display import Image as IPImage
from datetime import datetime

def compact_json(path):
    return json.dumps(json.loads(Path(path).read_text()), ensure_ascii=True, separators=(",", ":"))

# Load negative prompt from Cosmos audiovisual assets
neg_prompt_path = AUDIOVISUAL_ROOT / "assets/negative_prompts/image2video/neg_prompt.json"
neg_prompt = compact_json(neg_prompt_path) if neg_prompt_path.exists() else ""

# --- Inference configuration ---
I2V_SAMPLING = {
    "num_steps": 50,
    "guidance": 6.0,
    "shift": 3.0,
    "fps": 24,
    "num_frames": 193,      # 121 = ~5s  |  193 = ~8s
    "resolution": "480",    # "480" faster  |  "720" higher quality
    "aspect_ratio": "16,9",
    "seed": 42,              # change seed for motion variety
}
# --------------------------------

# Each run gets its own timestamped folder
RUN_I2V = datetime.now().strftime("%Y%m%d_%H%M%S")
I2V_RUN_DIR = OUTPUTS / "i2v" / RUN_I2V
I2V_RUN_DIR.mkdir(parents=True, exist_ok=True)

payload_path = I2V_RUN_DIR / "payload.json"
payload = {
    "model_mode": "image2video",
    "name": "sdg_i2v",
    "prompt": compact_json(SAMPLE_PROMPT_FILE), 
    "negative_prompt": neg_prompt,
    "enable_sound": False,
    **I2V_SAMPLING,
    "vision_path": str(AUGMENTED_IMAGE.resolve()),
}

payload_path.write_text(json.dumps(payload, indent=2) + "\n")
os.environ["SDG_I2V_PAYLOAD"] = str(payload_path)
os.environ["SDG_I2V_OUTPUT"] = str(I2V_RUN_DIR)
os.environ["SDG_I2V_RUN"] = RUN_I2V

print(f"Run:     {RUN_I2V}")
print(f"Payload: {payload_path}")
print(f"Output:  {I2V_RUN_DIR}")
print(json.dumps({k: payload[k] for k in ["model_mode", "num_steps", "guidance", "fps", "num_frames", "resolution"]}, indent=2))
print("Input image:")
display(IPImage(filename=str(AUGMENTED_IMAGE), width=480))

Generation starts here — this calls the Cosmos inference script and will take 2–4 minutes. Progress logs stream in the output as the model runs.

In [ ]:
%%bash
set -euo pipefail
cd "$COSMOS3_REPO"
CUDA_VISIBLE_DEVICES="$CUDA_VISIBLE_DEVICES" LD_LIBRARY_PATH= \
.venv/bin/torchrun \
  --nproc-per-node="$COSMOS3_NUM_GPUS" \
  --master-addr="$COSMOS3_MASTER_ADDR" \
  --master-port="$COSMOS3_MASTER_PORT" \
  -m cosmos_framework.scripts.inference \
  --parallelism-preset=throughput \
  -i "$SDG_I2V_PAYLOAD" \
  -o "$SDG_I2V_OUTPUT" \
  --checkpoint-path "$COSMOS3_CHECKPOINT_PATH" \
  --seed=0

Once generation finishes, run the cell below to find and play the output video inline.

**After running, here is what to look for:**
- Does the first frame match your input image?
- Does the motion match the action described in your prompt?
- Is the motion physically coherent — no teleporting subjects, flickering, or abrupt scene cuts?
- Would you keep this clip, try a different seed, adjust the parameters, or adjust the prompt?

In [ ]:
import base64, html as html_lib
from IPython.display import HTML, display

def display_video(path, width=720):
    data = base64.b64encode(Path(path).read_bytes()).decode("ascii")
    label = html_lib.escape(str(path))
    display(HTML(
        f'<video controls playsinline width="{width}" style="max-width:100%; background:#000">'
        f'<source src="data:video/mp4;base64,{data}" type="video/mp4"></video>'
        f'<div style="font-family:monospace;font-size:12px;margin-top:4px">{label}</div>'
    ))

expected = I2V_RUN_DIR / "sdg_i2v" / "vision.mp4"
if expected.exists():
    generated_video = expected
else:
    candidates = [p for p in I2V_RUN_DIR.rglob("*.mp4") if not p.name.endswith(("_preview.mp4", "_browser.mp4"))]
    generated_video = max(candidates, key=lambda p: p.stat().st_mtime) if candidates else None

if generated_video:
    print(f"Run: {RUN_I2V}")
    print(f"Generated: {generated_video} ({generated_video.stat().st_size // 1024} KB)")
    display_video(generated_video)
    os.environ["SDG_GENERATED_VIDEO"] = str(generated_video)
    meta = {
        "stage": "i2v", "run": RUN_I2V, "model": COSMOS3_CHECKPOINT,
        "source_image": str(AUGMENTED_IMAGE), "prompt_file": str(SAMPLE_PROMPT_FILE),
        "sampling": I2V_SAMPLING, "output": str(generated_video),
    }
    (I2V_RUN_DIR / "meta.json").write_text(json.dumps(meta, indent=2))
    print(f"Provenance saved: {I2V_RUN_DIR / 'meta.json'}")
else:
    print(f"No video found under {I2V_RUN_DIR} — run the inference cell above first.")

If the clip doesn't match your intent, try changing the parameters in `I2V_SAMPLING`, or update `SAMPLE_PROMPT_FILE` to a different scenario and rerun the payload and inference cells. Each run is saved to a timestamped folder under `assets/outputs/i2v/` so all previous results are preserved.

You can repeat the payload and inference cells with a different `seed` or a different prompt file to generate additional motion variants of the same image. Each variant expands your training dataset along the temporal axis.

---
## Part 3: Video Augmentation with Cosmos

So far, we've used Cosmos 3 to generate videos from scratch, based on a prompt and initial image. Cosmos 3 can also be used to augment existing video datasets. Cosmos3 Transfer re-renders a video's visual appearance — materials, lighting, surface textures, environment style — while aiming to preserve the coarse structure and motion from the source video. This is video-to-video (V2V) generation: a structural control signal extracted from the source video constrains the spatial layout, and the prompt drives the new appearance. Note that fine details and fast motion can still shift — outputs should be reviewed for structural fidelity.

It works by extracting a structural control signal from the source video and conditioning generation on that signal rather than on the pixels directly.

### 3.1 Transfer Controls

Five control modalities are available. Each extracts a different type of structural information from the source video:

| Control | Signal extracted | What it preserves | Best for |
|---|---|---|---|
| **Edge** | Canny edge map | Shapes, layout, fine structural detail | Material / texture changes, environment style |
| **Blur** | Low-frequency appearance | Colors, overall spatial layout | Subtle look changes, stylization |
| **Depth** | Monocular depth map | 3D structure, spatial scale, Z-ordering | Relighting, surface material changes |
| **Segmentation** | Semantic segment map | Object identity and region boundaries | Replacing backgrounds, object appearance swap |
| **WSM** | World-scenario map | High-level scene semantics | Broad scene-type transfer |

Edge is the most broadly applicable starting point — it preserves fine structural detail and allows significant appearance change. This notebook runs an edge transfer; the spec format is identical for all control types, changing only the control key.

**Transfer prompt best practices:**
- Describe the *target appearance*, not the source — the structure comes from the control signal, not the prompt
- Be specific about new conditions: `"nighttime, wet surfaces, sodium vapor streetlights"` works better than `"dark scene"`
- `guidance` (valid range 0–7) controls how strongly the generated video follows your **text prompt** — the appearance, materials, weather, and lighting you described
- `control_guidance` (valid range 0–10) controls how strongly the generated video follows the **control video** — how rigidly it enforces the Edge map's structure (vehicle silhouettes, lane boundaries, scene geometry). At `1.0` it's a no-op; pushing it higher enforces the source structure more strictly, at the cost of the model's freedom to deviate for the sake of the new appearance

These two are independent CFG knobs applied in separate forward passes — `guidance` pulls toward your prompt's appearance, `control_guidance` pulls toward the control signal's structure. Raise `guidance` if the appearance isn't changing enough; raise `control_guidance` if vehicles/edges are drifting or losing structural fidelity.

### 3.1.1 Generate a Transfer Prompt with AI

Unlike the I2V prompt in section 2.2.1 (which uses `nemotron-nano-vl-8b`, an **image-only** vision-language model), this step uses **`nemotron-3-nano-omni-30b-a3b-reasoning`** — a native omni model that accepts text, image, video, and audio in one architecture, with an explicit reasoning budget for working through what it sees before answering.

Video understanding genuinely needs a video-capable model here: the transfer prompt has to describe the target *appearance* while explicitly stating that motion, trajectories, and event timing are unchanged — which requires the model to actually perceive the motion across the clip, not just a single still frame.

This requires an `NVIDIA_API_KEY` set in your environment — see 2.2.1 if you haven't set that up yet. This cell loads it from `.env` itself, so it works standalone even if you haven't run 2.2.1's setup cell in this session.

A transfer prompt uses a rich schema for prompting - see [`assets/prompts/transfer_prompt_schema.json`](assets/prompts/transfer_prompt_schema.json). It should describe **only the target appearance**. Motion and scene structure come from the control signal extracted from the source video, not from the prompt — so the model is instructed to keep every motion/behavior field (`action`, `state_changes`, `relationship`, `location`, `relative_size`, `orientation`, `pose`, plus the `cinematography` camera fields) identical to the source video, and to vary only appearance, lighting, and environment. The schema also adds:
- `environmental_transformation` — the target condition, what's allowed to change, and an explicit `prohibited_changes` list
- `control_guidance` — which control modality is being used (`"Edge"` in this notebook) and how strictly it should be honored
- `negative_prompt` / `audio_description` — concepts to avoid and audio-preservation guidance

In [ ]:
import os
from pathlib import Path

# --- Configure ---
TRANSFER_DESCRIPTION = "Nighttime conditions with snowy sideroads, and streetlights as the primary light source."
# -----------------

# Locate the source video generated in Part 2 (same lookup used by the transfer config cell below)
_env_video = os.environ.get("SDG_GENERATED_VIDEO", "")
_transfer_source_video = Path(_env_video) if _env_video and Path(_env_video).exists() else None
if _transfer_source_video is None:
    _i2v_videos = sorted(
        (OUTPUTS / "i2v").rglob("vision.mp4"),
        key=lambda p: p.stat().st_mtime
    ) if (OUTPUTS / "i2v").exists() else []
    _transfer_source_video = _i2v_videos[-1] if _i2v_videos else None

if _transfer_source_video is None or not _transfer_source_video.exists():
    raise FileNotFoundError(f"No generated video found under {OUTPUTS / 'i2v'}. Run Part 2 first.")

print(f"Description:  {TRANSFER_DESCRIPTION}")
print(f"Source video: {_transfer_source_video}")


This sends the source video and your description to the model and waits for the target-appearance JSON prompt (non-streaming, since the reasoning model returns its full answer at once).

In [ ]:
import base64
import subprocess
from openai import OpenAI
from dotenv import load_dotenv

# --- Load NVIDIA_API_KEY from .env, overriding any stale/empty value already in the environment ---
_env_path = Path.cwd() / ".env"
_loaded = load_dotenv(_env_path, override=True)
NVIDIA_API_KEY = os.environ.get("NVIDIA_API_KEY", "")
print(f".env found at {_env_path}: {_env_path.exists()} | loaded: {_loaded} | key set: {bool(NVIDIA_API_KEY)}")
if not NVIDIA_API_KEY:
    raise EnvironmentError(
        "NVIDIA_API_KEY is not set. Edit the .env file and put your key there: NVIDIA_API_KEY=nvapi-... "
        "then restart the kernel or re-run this cell."
    )

# --- Reuse the NVIDIA client from 2.2.1 if it exists, otherwise create it here ---
try:
    client
except NameError:
    client = OpenAI(base_url="https://integrate.api.nvidia.com/v1", api_key=NVIDIA_API_KEY)

# nemotron-3-nano-omni is a native omni model (text/image/video/audio) with an explicit reasoning budget —
# unlike nemotron-nano-vl-8b (image-only), it can actually reason over the source video's motion and events.
NVIDIA_TRANSFER_MODEL = "nvidia/nemotron-3-nano-omni-30b-a3b-reasoning"

# --- Load the transfer prompt schema (see assets/prompts/transfer_prompt_schema.json) ---
TRANSFER_PROMPT_SCHEMA_FILE = ASSETS / "prompts" / "transfer_prompt_schema.json"
TRANSFER_PROMPT_SCHEMA = TRANSFER_PROMPT_SCHEMA_FILE.read_text()

# --- Instructions telling the model how to fill in that schema for a V2V transfer ---
SYSTEM_PROMPT_TRANSFER = f"""You are an expert at writing structured prompts for the Cosmos3 world foundation model for video-to-video (V2V) appearance transfer.

Given a source video (its structure and motion are preserved automatically via a control signal extracted from the video — not from this prompt) and a description of the desired new appearance, produce a structured JSON prompt in exactly this format:

{TRANSFER_PROMPT_SCHEMA}

Critical rules:
- Output ONLY valid JSON — no markdown fences, no preamble, no explanation after
- Describe ONLY the target appearance change: materials, lighting, weather, time of day, surface conditions
- Every field describing motion or behavior ("action", "state_changes", "relationship", "location", "relative_size", "orientation", "pose", "camera_motion", "framing", "camera_angle") must state that it is identical to the source video — do not invent new motion, behavior, or camera movement
- "background_setting" must include: 'The same structural layout and scene geometry as the source video' and 'All scene geometry, camera angle, and composition are preserved exactly'
- "environmental_transformation.prohibited_changes" must be a list of concrete things the model must not do to subjects, trajectories, or scene structure
- "control_guidance.control_type" must match the transfer control being used (this notebook uses "Edge")
- Add additional subject entries only if the video plausibly contains multiple distinct subject types (e.g. vehicles and pedestrians)"""


# --- Compress the video for this request only if it's too large for the API's payload limit ---
_MAX_PAYLOAD_BYTES = 26_214_400
_MAX_RAW_VIDEO_BYTES = int(_MAX_PAYLOAD_BYTES * 0.7)

_video_for_prompt = _transfer_source_video
if _video_for_prompt.stat().st_size > _MAX_RAW_VIDEO_BYTES:
    _compressed_path = _video_for_prompt.parent / f"{_video_for_prompt.stem}_vlm_compressed.mp4"
    subprocess.run(
        ["ffmpeg", "-y", "-i", str(_video_for_prompt), "-vf", "scale=480:-2",
         "-c:v", "libx264", "-crf", "30", "-preset", "fast", "-an", str(_compressed_path)],
        check=True, capture_output=True,
    )
    print(f"Source video ({_video_for_prompt.stat().st_size / 1024**2:.1f} MB) exceeds the VLM upload limit — "
          f"using a compressed copy ({_compressed_path.stat().st_size / 1024**2:.1f} MB) for prompt generation only. "
          f"The full-quality video is still used for the transfer inference step.")
    _video_for_prompt = _compressed_path

# --- Base64-encode the video and build the multimodal message (video + text instruction) ---
video_b64 = base64.b64encode(_video_for_prompt.read_bytes()).decode()
user_content = [
    {"type": "video_url", "video_url": {"url": f"data:video/mp4;base64,{video_b64}"}},
    {"type": "text", "text": f"Desired new appearance: {TRANSFER_DESCRIPTION}\n\nGenerate the Cosmos3 structured JSON transfer prompt for this scene."},
]

# --- Call the model. reasoning_budget isn't a standard OpenAI API field, so it goes through extra_body ---
REASONING_BUDGET = 16384

print(f"Generating with {NVIDIA_TRANSFER_MODEL} (this may take longer than the vision-only model due to reasoning)... ", end="", flush=True)
completion = client.chat.completions.create(
    model=NVIDIA_TRANSFER_MODEL,
    messages=[
        {"role": "system", "content": SYSTEM_PROMPT_TRANSFER},
        {"role": "user", "content": user_content},
    ],
    temperature=0.6,
    top_p=0.95,
    max_tokens=65536,
    stream=False,
    extra_body={"reasoning_budget": REASONING_BUDGET},
)

# --- Pull out the final answer (transfer_response_text is parsed as JSON in the next cell) ---
message = completion.choices[0].message
transfer_response_text = message.content or ""
print("done")
_reasoning = getattr(message, "reasoning_content", None)
if _reasoning:
    print(f"(model used {len(_reasoning)} chars of internal reasoning before answering)")


After the model responds, run this to parse the output as JSON, save it to `assets/prompts/llm_generated_transfer_prompt.json`, and render it for review. If parsing fails (the model added extra text), the raw output is saved for inspection.

In [ ]:
import json
import re
from IPython.display import JSON, display

raw = transfer_response_text.strip()
raw = re.sub(r"^```(?:json)?\s*\n?", "", raw)
raw = re.sub(r"\n?```\s*$", "", raw)
raw = raw.strip()

try:
    generated_transfer_prompt = json.loads(raw)
    output_path = ASSETS / "prompts" / "llm_generated_transfer_prompt.json"
    output_path.write_text(json.dumps(generated_transfer_prompt, indent=2))
    print(f"Saved: {output_path}")
    print("\nTo use this prompt, update transfer_prompt_path in the transfer config cell below:")
    print('  transfer_prompt_path = ASSETS / "prompts" / "llm_generated_transfer_prompt.json"')
    print()
except json.JSONDecodeError as e:
    print(f"Could not parse response as JSON: {e}")
    print("The model may have included extra text. Raw output saved to:")
    raw_path = ASSETS / "prompts" / "llm_generated_transfer_prompt_raw.txt"
    raw_path.write_text(transfer_response_text)
    print(f"  {raw_path}")
    print("\nRaw response:\n", transfer_response_text)


Alternatively, two sample transfer prompts are provided in `assets/prompts/` (`transfer_nighttime_snowy.json`, `transfer_nighttime_wet.json`), written in the same richer schema as `transfer_prompt_schema.json`. The next section uses the snowy sample by default — swap `transfer_prompt_path` there to point at your generated prompt, or a sample, before running inference.

### 3.2 Run Edge Transfer

The sample transfer prompt (`transfer_nighttime_snowy.json`) re-renders the scene under nighttime winter conditions — active snowfall, snow-covered road surfaces, and artificial lighting as the only light source. This is a high-value SDG variation: nighttime and adverse-weather conditions like snow are severely underrepresented in real-world intersection datasets.

**Model:** Transfer also uses **Cosmos3-Nano** by default. Switching to Cosmos3-Super produces sharper edge adherence and more photorealistic appearance results, at the cost of requiring multiple GPUs and longer generation time.

This configures the transfer: locates the video generated in Part 2, selects the edge control modality, and writes the spec file the inference script will read.

In [ ]:
import json
import socket
from datetime import datetime
from pathlib import Path


def free_local_port():
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
        s.bind(("127.0.0.1", 0))
        return str(s.getsockname()[1])


# --- Environment variables required by the %%bash inference cell below.
# Normally set by the Part 2.1 "Environment Setup" cell — set here too so Part 3
# also works standalone, without requiring Part 2 to have run first. ---
_COSMOS_ROOT = Path(os.environ.get("COSMOS_ROOT", "/localhome/local-sisimbimanzi/cosmos")).resolve()
_COSMOS3_REPO = Path(os.environ.get("COSMOS3_REPO", "/localhome/local-sisimbimanzi/cosmos/packages/cosmos-framework")).resolve()
os.environ.setdefault("CUDA_VISIBLE_DEVICES", "3")
os.environ.setdefault("COSMOS3_NUM_GPUS", "1")
os.environ.setdefault("COSMOS3_MASTER_ADDR", "127.0.0.1")
os.environ.setdefault("COSMOS3_MASTER_PORT", free_local_port())
os.environ.setdefault("HF_HOME", str(Path.home() / ".cache" / "huggingface"))
os.environ["COSMOS3_REPO"] = str(_COSMOS3_REPO)
os.environ.setdefault("COSMOS3_CHECKPOINT_PATH", "Cosmos3-Nano")
print(f"COSMOS3_REPO: {_COSMOS3_REPO} [{'OK' if _COSMOS3_REPO.exists() else 'MISSING'}]")
print(f"CUDA devices: {os.environ['CUDA_VISIBLE_DEVICES']} | Checkpoint: {os.environ['COSMOS3_CHECKPOINT_PATH']}")


def compact_json(path):
    return json.dumps(json.loads(Path(path).read_text()), ensure_ascii=True, separators=(",", ":"))


# Find the source video — use current session's I2V run if available
_env_video = os.environ.get("SDG_GENERATED_VIDEO", "")
source_video = Path(_env_video) if _env_video and Path(_env_video).exists() else None
if source_video is None:
    i2v_videos = sorted(
        (OUTPUTS / "i2v").rglob("vision.mp4"),
        key=lambda p: p.stat().st_mtime
    ) if (OUTPUTS / "i2v").exists() else []
    if i2v_videos:
        source_video = i2v_videos[-1]
    else:
        raise FileNotFoundError(f"No generated video found under {OUTPUTS / 'i2v'}. Run Part 2 first.")

print(f"Source video: {source_video}")

# Each transfer run gets its own timestamped folder
RUN_V2V = datetime.now().strftime("%Y%m%d_%H%M%S")
TRANSFER_RUN_DIR = OUTPUTS / "transfer_edge" / RUN_V2V
TRANSFER_RUN_DIR.mkdir(parents=True, exist_ok=True)

transfer_spec_path = TRANSFER_RUN_DIR / "spec.json"
transfer_prompt_path = ASSETS / "prompts" / "transfer_nighttime_snowy.json`"  #edit to point to your desired prompt
transfer_prompt = compact_json(transfer_prompt_path)
TRANSFER_NEG_PROMPT = ASSETS / "negative_prompt.json"

# --- Transfer configuration ---
transfer_spec = {
    "name": "sdg_transfer_edge",
    "model_mode": "video2video",
    "vision_path": str(source_video.resolve()),
    "prompt": transfer_prompt,
    "negative_prompt_file": str(TRANSFER_NEG_PROMPT),
    "edge": {"preset_edge_threshold": "medium"},
    "control_guidance": 3.0,
    "guidance": 4.0,
    "shift": 10.0,
    "num_steps": 45,
    "fps": 24,
    "num_frames": 193,
    "resolution": "480",
    "aspect_ratio": "16,9",
    "seed": 2026,
    "num_video_frames_per_chunk": 193,
    "num_conditional_frames": 1,
    "num_first_chunk_conditional_frames": 0,
    "share_vision_temporal_positions": True,
    "negative_metadata_mode": "none",
    "negative_prompt_keep_metadata": False,
    "emphasize_control_in_prompt": False,
}
# -------------------------------------------------------------------------------

transfer_spec_path.write_text(json.dumps(transfer_spec, indent=2) + "\n")
os.environ["SDG_TRANSFER_SPEC"] = str(transfer_spec_path)
os.environ["SDG_TRANSFER_OUTPUT"] = str(TRANSFER_RUN_DIR)

print(f"Run:           {RUN_V2V}")
print(f"Transfer spec: {transfer_spec_path}")
print(f"Output:        {TRANSFER_RUN_DIR}")
print(json.dumps({k: transfer_spec[k] for k in ["model_mode", "num_steps", "guidance", "shift", "fps", "num_frames", "resolution", "control_guidance"]}, indent=2))

Next, Cosmos3 transfer does the augmentation. Expect 18-20 minutes for an 8sec clip — progress logs will appear in the output.

In [ ]:
%%bash
set -euo pipefail
cd "$COSMOS3_REPO"
CUDA_VISIBLE_DEVICES="$CUDA_VISIBLE_DEVICES" LD_LIBRARY_PATH= \
.venv/bin/python -m cosmos_framework.scripts.inference \
  --parallelism-preset=latency \
  --no-use-torch-compile \
  -i "$SDG_TRANSFER_SPEC" \
  -o "$SDG_TRANSFER_OUTPUT" \
  --checkpoint-path "$COSMOS3_CHECKPOINT_PATH" \
  --seed 2026

The cell below displays the original generated video and the transferred version side by side. Both clips have identical motion — only the lighting and surface appearance differ.

**Before running — what to look for:**
- Did the appearance change to the conditions described in the transfer prompt?
- Does the transferred video retain the same motion and scene structure as the original?
- Are there areas where the control signal did not hold structure well?
- Would you keep this clip, increase `control_guidance` for tighter structure adherence, or try a different transfer control?

In [ ]:
expected_src = source_video
expected_trn = TRANSFER_RUN_DIR / "sdg_transfer_edge" / "vision.mp4"
if not expected_trn.exists():
    candidates = [p for p in TRANSFER_RUN_DIR.rglob("*.mp4") if not p.name.endswith(("_preview.mp4", "_browser.mp4"))]
    transfer_video_path = max(candidates, key=lambda p: p.stat().st_mtime) if candidates else None
else:
    transfer_video_path = expected_trn

source_video_path = expected_src if expected_src.exists() else None

if source_video_path and transfer_video_path:
    src_b64 = base64.b64encode(source_video_path.read_bytes()).decode()
    trn_b64 = base64.b64encode(transfer_video_path.read_bytes()).decode()
    display(HTML(f"""
<div style="display: flex; gap: 20px;">
  <div style="flex: 1;">
    <h3>Original Generated Video</h3>
    <video controls playsinline width="100%" style="background:#000">
      <source src="data:video/mp4;base64,{src_b64}" type="video/mp4">
    </video>
  </div>
  <div style="flex: 1;">
    <h3>Edge Transfer — Nighttime / Snowy</h3>
    <video controls playsinline width="100%" style="background:#000">
      <source src="data:video/mp4;base64,{trn_b64}" type="video/mp4">
    </video>
  </div>
</div>
<p style="font-size: 12px; font-family: monospace;">
  Run: {RUN_V2V} | Source: {source_video_path.name} | Transfer: {transfer_video_path.name}
</p>
"""))
    meta = {
        "stage": "v2v", "run": RUN_V2V, "model": COSMOS3_CHECKPOINT, "control": "edge",
        "source_video": str(source_video_path), "prompt_file": str(transfer_prompt_path),
        "spec": transfer_spec, "output": str(transfer_video_path),
    }
    (TRANSFER_RUN_DIR / "meta.json").write_text(json.dumps(meta, indent=2))
    print(f"Provenance saved: {TRANSFER_RUN_DIR / 'meta.json'}")
else:
    if not source_video_path:
        print("Source video not found — run Part 2 inference first.")
    if not transfer_video_path:
        print("Transfer video not found — run the transfer inference cell above first.")

If the transfer result doesn't match the target appearance, try adjusting `guidance` to follow the prompt more closely, swap `transfer_prompt_path` to a different prompt, or try a different control modality. Rerun the configuration and transfer cells — each run is saved to a timestamped folder under `assets/outputs/transfer_edge/` so all previous results are preserved.

---
## Review

You completed the generation-and-augmentation stage of an SDG pipeline, starting from a single seed image:

- **Qwen Image Edit (I2I)** changed the scene's appearance conditions → image variants with different environmental context
- **Cosmos3 I2V** generated motion from the augmented image → video clips with temporal dynamics
- **Cosmos3 Transfer (V2V)** re-rendered the video under new appearance conditions → appearance variants without re-shooting

The dataset scaling effect compounds across stages:

```
1 seed image
  × N edit prompts    → N augmented images
  × M motion prompts  → N × M generated videos
  × K transfer types  → N × M × K total training samples
```

Provenance for each stage is saved in `assets/outputs/` (`i2i_meta.json`, `i2v_meta.json`, `v2v_meta.json`), recording the source, prompt, model, and parameters used.

**Next steps to explore:**

- **Try other Cosmos3 models** — this notebook uses Cosmos3-Nano (fast, single GPU, good for iteration). Two other variants are available:
  - **Cosmos3-Super** (32B parameters, 2 GPUs) — significantly higher visual quality and motion realism; recommended for final dataset generation once your prompts are dialed in. Switch via `COSMOS3_CHECKPOINT_PATH=Cosmos3-Super` and `COSMOS3_NUM_GPUS=2`
  - **Cosmos3-Edge** (4B parameters, 1 GPU) - a lightweight model optimized for edge device deployment; useful when generating data to train models that will run on constrained hardware
- **More Qwen edits** — vary weather, time of day, sensor type, or object count to expand image-level diversity
- **More motion variants** — change `seed` in the I2V payload, swap between the three scenario prompts, edit a prompt manually to describe a different event, or use the LLM section (2.2.1) to generate a new one from a plain-language description
- **Other transfer controls** — try `depth` for geometry-grounded relighting, `seg` for background replacement, or `blur` for style-level changes
- **Combine with annotation** — pair generated videos with bounding boxes, depth maps, or segmentation masks for fully labeled synthetic data